In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Milestone 3: Qwen3-VL Zero-shot VideoQA

This notebook runs a deterministic zero-shot baseline on the
NExT-QA pilot subset using Qwen3-VL-2B-Instruct.

Experiment:
- Dataset: NExT-QA pilot validation subset
- Videos: 12
- Questions: 48
- Sampling: uniform, 8-frame smoke test followed by 16-frame pilot
- Decoding: greedy

In [10]:
%cd /content

!rm -rf /content/LongVideoGuard

!git clone \
  https://github.com/Maxineee720/LongVideoGuard.git \
  /content/LongVideoGuard

/content
Cloning into '/content/LongVideoGuard'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 84 (delta 21), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 34.06 KiB | 17.03 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [11]:
!ls /content/LongVideoGuard

AGENTS.md	     data     notebooks       requirements-colab.txt  tests
apply_milestone3.py  docs     pyproject.toml  scripts
configs		     LICENSE  README.md       src


In [12]:
!rsync -a \
  /content/longvideoguard_data/ \
  /content/LongVideoGuard/data/

!rm -rf /content/longvideoguard_data

In [13]:
%cd /content/LongVideoGuard

/content/LongVideoGuard


In [14]:
!ls
!find data/raw/nextqa/videos -type f | wc -l
!wc -l data/processed/nextqa/pilot_val.jsonl

AGENTS.md	     data     notebooks       requirements-colab.txt  tests
apply_milestone3.py  docs     pyproject.toml  scripts
configs		     LICENSE  README.md       src
12
48 data/processed/nextqa/pilot_val.jsonl


In [15]:
!pip install -U pip
!pip install -e ".[dev,vlm]"

Obtaining file:///content/LongVideoGuard
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 150.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 183.9 MB/s  0:00:00
  Building editable for longvideoguard (pyproject.toml) ... done
  Created wheel for longvideoguard: filename=longvideoguard-0.1.0-0.editable-py3-none-any.whl size=4710 sha256=28cc042dcf91565a1d17318b3c5b39915fd4f33ee0a50cc0590f7f9e57786117
  Stored in directory: /tmp/pip-ephem-wheel-cache-3ll1ajsy/wheels/1b/83/33/0482add59765ec931bb7b522392725f8193ca2d9f2da76e277
Successfully built longvideoguard
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [longvideoguard]m [av]


In [16]:
!pytest

..........................                                               [100%]
26 passed in 0.22s


In [17]:
!PYTHONPATH=src python -m longvideoguard.cli validate-videos \
  data/processed/nextqa/pilot_val.jsonl \
  data/raw/nextqa/videos

{
  "num_unique_videos": 12,
  "status_counts": {
    "ok": 12
  },
  "all_videos_ready": true
}
Validation report: 
/content/LongVideoGuard/data/processed/nextqa/pilot_video_validation.json


In [18]:
import torch
import transformers

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16 supported: True
PyTorch: 2.11.0+cu128
Transformers: 5.13.1


In [19]:
%cd /content/LongVideoGuard

!ls -lh scripts/run_nextqa_zero_shot.py
!ls -lh scripts/inspect_prediction_file.py

/content/LongVideoGuard
-rw-r--r-- 1 root root 7.6K Jul 18 19:07 scripts/run_nextqa_zero_shot.py
-rw-r--r-- 1 root root 1.2K Jul 18 19:07 scripts/inspect_prediction_file.py


In [20]:
!python scripts/run_nextqa_zero_shot.py \
  data/processed/nextqa/pilot_val.jsonl \
  data/raw/nextqa/videos \
  --num-frames 8 \
  --limit 1 \
  --output outputs/predictions/smoke_qwen3vl2b_frames8.jsonl \
  --overwrite

Model: Qwen/Qwen3-VL-2B-Instruct
Manifest rows selected: 1
Frame budget: 8
Output: /content/LongVideoGuard/outputs/predictions/smoke_qwen3vl2b_frames8.jsonl
preprocessor_config.json: 100% 390/390 [00:00<00:00, 2.03MB/s]
chat_template.json: 100% 5.50k/5.50k [00:00<00:00, 15.7MB/s]
config.json: 100% 1.50k/1.50k [00:00<00:00, 6.62MB/s]
tokenizer_config.json: 100% 10.9k/10.9k [00:00<00:00, 32.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 118MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 138MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 2.08MB/s]

model.safetensors: downloading bytes:   5% 197M/4.26G [00:01<00:20, 199MB/s, 13.1MB/s  ]  
model.safetensors: downloading bytes:   7% 294M/4.26G [00:01<00:14, 280MB/s, 22.4MB/s  ]
model.safetensors: downloading bytes:  11% 455M/4.26G [00:02<00:08, 457MB/s, 34.7MB/s  ] ]
model.safetensors: downloading bytes:  14% 580M/4.26G [00:02<00:07, 515MB/s, 47.3MB/s  ] ]
model.saf

In [21]:
from pathlib import Path

path = Path("src/longvideoguard/qwen3vl_runner.py")
text = path.read_text(encoding="utf-8")

old_video = '''                    {
                        "type": "video",
                        "video": path.as_uri(),
                    },'''

new_video = '''                    {
                        "type": "video",
                        "url": str(path),
                    },'''

old_processor = '''            return_dict=True,
            return_tensors="pt",
            num_frames=num_frames,
            fps=None,
        )
        inputs = inputs.to(self.model.device)'''

new_processor = '''            return_dict=True,
            processor_kwargs={
                "text_kwargs": {
                    "return_tensors": "pt",
                },
                "videos_kwargs": {
                    "do_sample_frames": True,
                    "num_frames": num_frames,
                },
            },
        )
        inputs.pop("token_type_ids", None)
        inputs = inputs.to(self.model.device)'''

if old_video not in text:
    raise RuntimeError("没有找到旧的视频输入代码，请检查文件内容。")

if old_processor not in text:
    raise RuntimeError("没有找到旧的 Processor 代码，请检查文件内容。")

text = text.replace(old_video, new_video)
text = text.replace(old_processor, new_processor)

path.write_text(text, encoding="utf-8")

print("Patched:", path)

Patched: src/longvideoguard/qwen3vl_runner.py


In [22]:
!grep -n -A8 -B3 '"type": "video"' \
  src/longvideoguard/qwen3vl_runner.py

83-                "role": "user",
84-                "content": [
85-                    {
86:                        "type": "video",
87-                        "url": str(path),
88-                    },
89-                    {
90-                        "type": "text",
91-                        "text": prompt,
92-                    },
93-                ],
94-            }


In [23]:
!grep -n -A15 -B3 "processor_kwargs" \
  src/longvideoguard/qwen3vl_runner.py

99-            tokenize=True,
100-            add_generation_prompt=True,
101-            return_dict=True,
102:            processor_kwargs={
103-                "text_kwargs": {
104-                    "return_tensors": "pt",
105-                },
106-                "videos_kwargs": {
107-                    "do_sample_frames": True,
108-                    "num_frames": num_frames,
109-                },
110-            },
111-        )
112-        inputs.pop("token_type_ids", None)
113-        inputs = inputs.to(self.model.device)
114-
115-        torch = self._torch
116-        using_cuda = torch.cuda.is_available()
117-        if using_cuda:


In [24]:
!pytest

..........................                                               [100%]
26 passed in 0.19s


In [25]:
!python scripts/run_nextqa_zero_shot.py \
  data/processed/nextqa/pilot_val.jsonl \
  data/raw/nextqa/videos \
  --num-frames 8 \
  --limit 1 \
  --output outputs/predictions/smoke_qwen3vl2b_frames8.jsonl \
  --overwrite

Model: Qwen/Qwen3-VL-2B-Instruct
Manifest rows selected: 1
Frame budget: 8
Output: /content/LongVideoGuard/outputs/predictions/smoke_qwen3vl2b_frames8.jsonl
Loading weights: 100% 625/625 [00:01<00:00, 589.00it/s]
[1/1] RUN  7453733046:10
  output='' parsed=None gold=B error='ValueError: `num_frames` and `fps` are mutually exclusive arguments, please use only one!'
New predictions written: 1


In [26]:
from pathlib import Path

path = Path("src/longvideoguard/qwen3vl_runner.py")
text = path.read_text(encoding="utf-8")

old = """        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageTextToText.from_pretrained(
"""

new = """        self.processor = AutoProcessor.from_pretrained(model_name)

        # Transformers currently gives the Qwen3-VL video processor a
        # default fps value. We use a fixed num_frames budget instead,
        # so fps must be explicitly disabled to avoid passing both.
        self.processor.video_processor.fps = None

        self.model = AutoModelForImageTextToText.from_pretrained(
"""

if "self.processor.video_processor.fps = None" in text:
    print("Fix already applied.")
elif old not in text:
    raise RuntimeError("Could not find the expected processor initialization code.")
else:
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("Applied fixed-frame sampling compatibility fix.")

Applied fixed-frame sampling compatibility fix.


In [27]:
!grep -n -A8 -B3 "video_processor.fps" \
  src/longvideoguard/qwen3vl_runner.py

59-        # Transformers currently gives the Qwen3-VL video processor a
60-        # default fps value. We use a fixed num_frames budget instead,
61-        # so fps must be explicitly disabled to avoid passing both.
62:        self.processor.video_processor.fps = None
63-
64-        self.model = AutoModelForImageTextToText.from_pretrained(
65-            model_name,
66-            **model_kwargs,
67-        )
68-        self.model.eval()
69-
70-    def generate_video_answer(


In [28]:
!pytest

..........................                                               [100%]
26 passed in 0.20s


In [29]:
!python scripts/run_nextqa_zero_shot.py \
  data/processed/nextqa/pilot_val.jsonl \
  data/raw/nextqa/videos \
  --num-frames 8 \
  --limit 1 \
  --output outputs/predictions/smoke_qwen3vl2b_frames8.jsonl \
  --overwrite

Model: Qwen/Qwen3-VL-2B-Instruct
Manifest rows selected: 1
Frame budget: 8
Output: /content/LongVideoGuard/outputs/predictions/smoke_qwen3vl2b_frames8.jsonl
Loading weights: 100% 625/625 [00:01<00:00, 589.39it/s]
[1/1] RUN  7453733046:10
  output='B' parsed='B' gold=B error=None
New predictions written: 1


In [30]:
!python scripts/inspect_prediction_file.py \
  outputs/predictions/smoke_qwen3vl2b_frames8.jsonl

rows: 1
valid predictions: 1/1
correct: 1/1
errors: 0/1
7453733046:10: raw='B', pred='B', gold=B, correct=True


In [31]:
!python scripts/run_nextqa_zero_shot.py \
  data/processed/nextqa/pilot_val.jsonl \
  data/raw/nextqa/videos \
  --num-frames 16 \
  --dtype float16 \
  --output outputs/predictions/nextqa_qwen3vl2b_zero_shot_frames16.jsonl \
  --overwrite

Model: Qwen/Qwen3-VL-2B-Instruct
Manifest rows selected: 48
Frame budget: 16
Output: /content/LongVideoGuard/outputs/predictions/nextqa_qwen3vl2b_zero_shot_frames16.jsonl
Loading weights: 100% 625/625 [00:01<00:00, 500.80it/s]
[1/48] RUN  7453733046:10
  output='B' parsed='B' gold=B error=None
[2/48] RUN  7453733046:11
  output='A' parsed='A' gold=A error=None
[3/48] RUN  7453733046:3
  output='A' parsed='A' gold=A error=None
[4/48] RUN  7453733046:6
  output='A' parsed='A' gold=A error=None
[5/48] RUN  3375218204:0
  output='D' parsed='D' gold=D error=None
[6/48] RUN  3375218204:4
  output='C' parsed='C' gold=C error=None
[7/48] RUN  3375218204:5
  output='C' parsed='C' gold=E error=None
[8/48] RUN  3375218204:6
  output='E' parsed='E' gold=A error=None
[9/48] RUN  3372023610:7
  output='E' parsed='E' gold=A error=None
[10/48] RUN  3372023610:1
  output='D' parsed='D' gold=C error=None
[11/48] RUN  3372023610:2
  output='A' parsed='A' gold=B error=None
[12/48] RUN  3372023610:6
  outp